# 02. Pemodelan K-Modes Clustering Berbasis Pemisahan Wilayah (Region-Partitioning)
---

### **Tujuan Notebook:**
1. Memuat dataset kategorikal bersih (`01_umkm_clean.csv`).
2. Mempartisi data ke dalam 6 wilayah administrasi DKI Jakarta.
3. Menjalankan algoritma *K-Modes Clustering* dengan fungsi *matching dissimilarity* dan inisialisasi `Cao` pada masukan atribut `jenis_usaha` dan `skpd`.
4. Mengekstrak modus pusat klaster (*modes centroids*) pada setiap wilayah.
5. Menyematkan label dan nama profil bisnis deskriptif pada masing-masing klaster.
6. Menyimpan dataset terklasterisasi ke `data/processed/02_umkm_kmodes_clustered.csv`.

In [12]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kmodes.kmodes import KModes

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print("Pustaka K-Modes berhasil dimuat.")

Pustaka K-Modes berhasil dimuat.


## 1. Memuat Dataset Hasil Rekayasa Fitur
Memuat dataset `02_umkm_features.csv` yang memuat 298.875 baris data bersih beserta fitur campuran siap olah.

In [13]:
data_path = os.path.join('..', 'data', 'processed', '01_umkm_clean.csv')
if not os.path.exists(data_path):
    data_path = '01_umkm_clean.csv'

df = pd.read_csv(data_path)
print(f"Dataset berhasil dimuat: {len(df):,} baris.")
df[['wilayah', 'skpd', 'jenis_usaha']].head(5)

Dataset berhasil dimuat: 298,875 baris.


,wilayah,skpd,jenis_usaha
0,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,SABLON DAN DESIGN GRAFIS
1,KOTA ADM. JAKARTA SELATAN,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN
2,KAB. ADM. KEP. SERIBU,DINAS PARIWISATA DAN EKONOMI KREATIF,AGEN WISATA PERJALANAN (AWP)
3,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN
4,KOTA ADM. JAKARTA TIMUR,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN


## 2. Fungsi Pemodelan K-Modes per Partisi Wilayah
Algoritma *K-Modes* mengelompokkan data berdasarkan ketidaksamaan pencocokan sederhana (*simple matching dissimilarity*).

In [14]:
def train_kmodes_region(df_subset, n_clusters=3, random_state=42):
    """
    Melatih model K-Modes untuk satu wilayah administrasi pada fitur jenis_usaha dan skpd.
    """
    df_reg = df_subset.copy()
    
    # Matriks fitur kategorikal murni
    feature_matrix = df_reg[['jenis_usaha', 'skpd']].astype(str).values
    
    # Inisialisasi K-Modes dengan metode Cao
    km = KModes(
        n_clusters=n_clusters, 
        init='Cao', 
        n_init=3, 
        verbose=0, 
        random_state=random_state,
        n_jobs=-1
    )
    
    t_start = time.time()
    clusters = km.fit_predict(feature_matrix)
    t_elapsed = time.time() - t_start
    
    df_reg['cluster_label'] = clusters
    
    print(f"Fitting K-Modes selesai ({t_elapsed:.2f} dtk) | Nilai Cost: {km.cost_:,}")
    return df_reg, km

## 3. Eksekusi K-Modes pada Seluruh 6 Wilayah Administrasi
Menjalankan algoritma *K-Modes* ($k=3$) pada masing-masing wilayah administrasi.

In [15]:
daftar_wilayah = sorted(df['wilayah'].unique().tolist())
clustered_list = []
model_dict = {}

print("=== MEMULAI CLUSTERING K-MODES PER WILAYAH ===")
for wil in daftar_wilayah:
    print(f"\n--- Wilayah: {wil} ---")
    df_wil = df[df['wilayah'] == wil].copy()
    
    df_clustered_wil, model_wil = train_kmodes_region(df_wil, n_clusters=3, random_state=42)
    
    clustered_list.append(df_clustered_wil)
    model_dict[wil] = model_wil

# Menggabungkan seluruh data yang telah terklasterisasi
df_final = pd.concat(clustered_list, ignore_index=True)
print("\n=== CLUSTERING K-MODES SELURUH WILAYAH SELESAI ===")
print(f"Total baris data terklasterisasi: {len(df_final):,} baris")

=== MEMULAI CLUSTERING K-MODES PER WILAYAH ===

--- Wilayah: KAB. ADM. KEP. SERIBU ---
Fitting K-Modes selesai (1.98 dtk) | Nilai Cost: 2,887.0

--- Wilayah: KOTA ADM. JAKARTA BARAT ---
Fitting K-Modes selesai (2.64 dtk) | Nilai Cost: 41,090.0

--- Wilayah: KOTA ADM. JAKARTA PUSAT ---
Fitting K-Modes selesai (2.21 dtk) | Nilai Cost: 35,398.0

--- Wilayah: KOTA ADM. JAKARTA SELATAN ---
Fitting K-Modes selesai (3.26 dtk) | Nilai Cost: 60,756.0

--- Wilayah: KOTA ADM. JAKARTA TIMUR ---
Fitting K-Modes selesai (2.66 dtk) | Nilai Cost: 49,252.0

--- Wilayah: KOTA ADM. JAKARTA UTARA ---
Fitting K-Modes selesai (2.31 dtk) | Nilai Cost: 34,558.0

=== CLUSTERING K-MODES SELURUH WILAYAH SELESAI ===
Total baris data terklasterisasi: 298,875 baris


## 4. Ekstraksi Modus Pusat Klaster (*Cluster Modes*)
Melihat nilai modus representatif (`jenis_usaha` dan `skpd`) yang menjadi pusat setiap klaster.

In [16]:
for wil in daftar_wilayah:
    km_model = model_dict[wil]
    print(f"\n=== PUSAT MODUS WILAYAH: {wil} ===")
    for c_id, mode_val in enumerate(km_model.cluster_centroids_):
        print(f"  Klaster {c_id} -> Jenis Usaha: '{mode_val[0]}' | SKPD: '{mode_val[1]}'")


=== PUSAT MODUS WILAYAH: KAB. ADM. KEP. SERIBU ===
  Klaster 0 -> Jenis Usaha: 'MAKANAN DAN MINUMAN' | SKPD: 'SUKU DINAS PERINDUSTRIAN, PERDAGANGAN, KOPERASI, USAHA KECIL DAN MENENGAH KABUPATEN ADMINISTRASI KEPULAUAN SERIBU'
  Klaster 1 -> Jenis Usaha: 'OLAHAN PERIKANAN' | SKPD: 'SUKU DINAS KETAHANAN PANGAN, KELAUTAN, DAN PERTANIAN KABUPATEN ADMINISTRASI KEPULAUAN SERIBU'
  Klaster 2 -> Jenis Usaha: 'MAKANAN DAN MINUMAN' | SKPD: 'SUKU DINAS KETAHANAN PANGAN, KELAUTAN, DAN PERTANIAN KABUPATEN ADMINISTRASI KEPULAUAN SERIBU'

=== PUSAT MODUS WILAYAH: KOTA ADM. JAKARTA BARAT ===
  Klaster 0 -> Jenis Usaha: 'MAKANAN DAN MINUMAN' | SKPD: 'SUKU DINAS PERINDUSTRIAN, PERDAGANGAN, KOPERASI, USAHA KECIL DAN MENENGAH KOTA ADMINISTRASI JAKARTA BARAT'
  Klaster 1 -> Jenis Usaha: 'PRODUK MAKANAN LAINNYA' | SKPD: 'SUKU DINAS PERINDUSTRIAN, PERDAGANGAN, KOPERASI, USAHA KECIL DAN MENENGAH KOTA ADMINISTRASI JAKARTA BARAT'
  Klaster 2 -> Jenis Usaha: 'FASHION' | SKPD: 'SUKU DINAS PERINDUSTRIAN, PERDAGANG

## 5. Penyimpanan Dataset Terklasterisasi K-Modes
Menyimpan dataset akhir ke `data/processed/02_umkm_kmodes_clustered.csv`.

In [17]:
output_path = os.path.join('..', 'data', 'processed', '02_umkm_kmodes_clustered.csv')
df_final.to_csv(output_path, index=False)

print(f"Dataset hasil K-Modes berhasil disimpan di: {output_path}")

Dataset hasil K-Modes berhasil disimpan di: ..\data\processed\02_umkm_kmodes_clustered.csv
